# `get_subsample_mhc` — Refatorado
**GT1 NEMPA · nempa-boltz**  
Geração do subconjunto MHC-peptídeo para finetuning do Boltz1.

---

### Purpose
Seleciona, a partir do manifesto RCSB completo, apenas as estruturas que contêm
um complexo MHC-peptídeo válido (exatamente 1 interface MHC-α ↔ peptídeo),
atualiza a máscara do `.npz` para excluir cadeias irrelevantes e gera os
manifestos de treino/validação prontos para uso pelo `BoltzTrainingDataModule`.

### O bug corrigido
O notebook original tentava `sample['protein_chains'][0]` sobre um registro do
`TCR3d_data.csv`, mas esse campo **não existe** nesse CSV.  
A solução adotada usa **alinhamento de sequência** contra o template **1A1M**
(cadeia MHC-α de referência) para identificar a cadeia correta em cada estrutura,
conforme decisão do Action Plan de 20/04/2026.

### Splits temporais
| Split | Intervalo | Critério |
|-------|-----------|---------|
| **train** | até 30/09/2021 | < 2021-09-30 |
| **val**   | 30/09/2021 – 13/01/2023 | [2021-09-30, 2023-01-13) |
| **test**  | a partir de 13/01/2023 | ≥ 2023-01-13 |

### Dependencies
```
pip install numpy pandas biopython
```
Bibliotecas padrão do ambiente `boltz_env` (numpy, pandas) mais **Biopython**
para alinhamento de sequência (pairwise2 / PairwiseAligner).

### Inputs / Outputs
- **Input:** `mhc_data/TCR3d_data.csv`, `rcsb_processed_targets/manifest.json`,
  `rcsb_processed_targets/structures/*.npz`, `rcsb_processed_msa/*.npz`
- **Output:** `mhc_samples/manifest.json`, `mhc_samples/structures/*.npz`,
  `mhc_samples/msa/*.npz`, `mhc_samples/validation_ids.txt`,
  `mhc_data/train_templated.txt`, `mhc_data/val_templated.txt`

### Limitations
- Assume MHC classe I (cadeia única α + β2m). MHC-II (αβ) não é tratado.
- Requer que o manifesto RCSB já tenha pré-processado os `.npz`.
- Sequência template 1A1M hardcoded; atualize `MHC_ALPHA_TEMPLATE_SEQ` para outras referências.


## 1. Imports & Configuração

In [1]:
import json
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Biopython — alinhamento de sequência
try:
    from Bio import Align
    from Bio.Align import substitution_matrices
    HAS_BIO = True
except ImportError:
    HAS_BIO = False
    warnings.warn(
        "Biopython não instalado. Execute: pip install biopython\n"
        "Fallback: identificação por comprimento de cadeia (menos robusto)."
    )

print("Imports OK | Biopython disponível:", HAS_BIO)

Imports OK | Biopython disponível: True


## 2. Constantes & Caminhos

In [2]:
# ── Caminhos ──────────────────────────────────────────────────────────────
RCSB_DIR        = Path("rcsb_processed_targets")
RCSB_MSA_DIR    = Path("rcsb_processed_msa")
MHC_DATA_DIR    = Path("mhc_data")
TARGET_DIR      = Path("mhc_samples")

# ── Splits temporais ──────────────────────────────────────────────────────
TRAIN_CUTOFF = pd.Timestamp("2021-09-30")
VAL_CUTOFF   = pd.Timestamp("2023-01-13")

# ── Sequência MHC-α do template 1A1M (HLA-B*53:01, domínios α1+α2) ────────
# Fonte: PDB 1A1M cadeia A. Usada como ground truth de alinhamento.
# Para trocar o template: substitua esta string.
MHC_ALPHA_TEMPLATE_SEQ = (
    "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLR"
    "GYYNQSEAGSHIIQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALNEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLR"
    "RYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRY"
    "TCHVQHEGLPKPLTLRWEPSSQSTIPIVGIVAGLAVLAVVALGIGLFSVGSNRTARDPPKTHLL"
)

# Limiar de identidade de sequência para aceitar cadeia como MHC-α
MHC_IDENTITY_THRESHOLD = 0.30   # 30% — conservador para capturar HLA variantes

# Comprimento mínimo para candidato a MHC-α (filtra peptídeos curtos)
MHC_ALPHA_MIN_LEN = 80

print("Caminhos configurados.")
print(f"  RCSB_DIR   : {RCSB_DIR}")
print(f"  TARGET_DIR : {TARGET_DIR}")
print(f"  Train até  : {TRAIN_CUTOFF.date()}")
print(f"  Val até    : {VAL_CUTOFF.date()}")

Caminhos configurados.
  RCSB_DIR   : rcsb_processed_targets
  TARGET_DIR : mhc_samples
  Train até  : 2021-09-30
  Val até    : 2023-01-13


## 3. Funções Auxiliares — Alinhamento de Sequência

In [3]:
# Mapa de inteiros → aminoácidos (padrão Boltz1 / RCSB processado)
INT_TO_AA = list("ACDEFGHIKLMNPQRSTVWY")  # 20 AAs canônicos, índice 0–19

def decode_sequence(res_type_array: np.ndarray) -> str:
    """Converte array de inteiros res_type → string de sequência 1-letra."""
    return "".join(
        INT_TO_AA[r] if 0 <= r < len(INT_TO_AA) else "X"
        for r in res_type_array
    )


def sequence_identity_biopython(seq_a: str, seq_b: str) -> float:
    """
    Retorna identidade de sequência (0.0–1.0) entre seq_a e seq_b
    usando alinhamento local com BLOSUM62 via Biopython PairwiseAligner.
    """
    if not seq_a or not seq_b:
        return 0.0
    aligner = Align.PairwiseAligner()
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    aligner.mode = "local"
    alignments = aligner.align(seq_a, seq_b)
    try:
        aln = next(iter(alignments))
        # Identidade = matches / comprimento do alinhamento
        aligned_a, aligned_b = aln[0], aln[1]
        matches = sum(a == b for a, b in zip(aligned_a, aligned_b) if a != "-" and b != "-")
        aln_len = max(len(seq_a), len(seq_b))
        return matches / aln_len if aln_len > 0 else 0.0
    except StopIteration:
        return 0.0


def sequence_identity_simple(seq_a: str, seq_b: str) -> float:
    """Fallback sem Biopython: identidade por comprimento apenas."""
    shorter = min(len(seq_a), len(seq_b))
    longer  = max(len(seq_a), len(seq_b))
    if longer == 0:
        return 0.0
    # Heurística: cadeias com 80–400 resíduos são candidatas a MHC-α
    ratio = shorter / longer
    return ratio if MHC_ALPHA_MIN_LEN <= shorter <= 400 else 0.0


def sequence_identity(seq_a: str, seq_b: str) -> float:
    if HAS_BIO:
        return sequence_identity_biopython(seq_a, seq_b)
    return sequence_identity_simple(seq_a, seq_b)


def is_peptide_sequence(seq: str, pep_seq: str, max_len: int = 25) -> bool:
    """
    Retorna True se a cadeia é o peptídeo antigênico esperado.
    Critério: comprimento ≤ max_len e sequência coincide com pep_seq do CSV.
    """
    if len(seq) > max_len:
        return False
    return seq == pep_seq or pep_seq in seq


print("Funções de alinhamento definidas.")

Funções de alinhamento definidas.


## 4. Carregar TCR3d_data.csv e Definir Splits

In [4]:
df = pd.read_csv(MHC_DATA_DIR / "TCR3d_data.csv")
df["PDB ID"]       = df["PDB ID"].astype(str).str.lower().str.strip()
df["Release date"] = pd.to_datetime(df["Release date"])

# Splits temporais
train_df = df[df["Release date"] < TRAIN_CUTOFF]
val_df   = df[(df["Release date"] >= TRAIN_CUTOFF) & (df["Release date"] < VAL_CUTOFF)]
test_df  = df[df["Release date"] >= VAL_CUTOFF]

train_ids = train_df["PDB ID"].dropna().tolist()
val_ids   = val_df["PDB ID"].dropna().tolist()
test_ids  = test_df["PDB ID"].dropna().tolist()

print(f"Total no CSV  : {len(df)}")
print(f"Train (< 2021-09-30)  : {len(train_ids)}")
print(f"Val   (< 2023-01-13)  : {len(val_ids)}")
print(f"Test  (>= 2023-01-13) : {len(test_ids)}")
print()
print("Primeira entrada (template 1A1M):")
train_df[train_df["PDB ID"] == "1a1m"][[
    "PDB ID", "MHC allele", "Species", "Peptide*", "Resolution", "Release date"
]]

Total no CSV  : 1452
Train (< 2021-09-30)  : 1082
Val   (< 2023-01-13)  : 130
Test  (>= 2023-01-13) : 240

Primeira entrada (template 1A1M):


,PDB ID,MHC allele,Species,Peptide*,Resolution,Release date
0,1a1m,HLA-B*53,Human,TPYDINQML,2.3,1998-04-07


## 5. Salvar Listas de IDs (train/val)

In [5]:
# Salva apenas uma vez; não é necessário re-rodar
MHC_DATA_DIR.mkdir(parents=True, exist_ok=True)

with open(MHC_DATA_DIR / "train_templated.txt", "w") as f:
    f.write("\n".join(train_ids))

with open(MHC_DATA_DIR / "val_templated.txt", "w") as f:
    f.write("\n".join(val_ids))

with open(MHC_DATA_DIR / "test_templated.txt", "w") as f:
    f.write("\n".join(test_ids))

print("Arquivos salvos:")
print(f"  {MHC_DATA_DIR}/train_templated.txt  ({len(train_ids)} IDs)")
print(f"  {MHC_DATA_DIR}/val_templated.txt    ({len(val_ids)} IDs)")
print(f"  {MHC_DATA_DIR}/test_templated.txt   ({len(test_ids)} IDs)")

Arquivos salvos:
  mhc_data/train_templated.txt  (1082 IDs)
  mhc_data/val_templated.txt    (130 IDs)
  mhc_data/test_templated.txt   (240 IDs)


## 6. Carregar Manifesto RCSB Completo

In [6]:
with open(RCSB_DIR / "manifest.json") as f:
    rcsb_data = json.load(f)

# Dict para lookup rápido por PDB ID
manifest_dict = {sample["id"]: sample for sample in rcsb_data}

print(f"Entradas no manifesto RCSB: {len(rcsb_data)}")
print(f"IDs de treino presentes   : {sum(i in manifest_dict for i in train_ids)}")
print(f"IDs de val presentes      : {sum(i in manifest_dict for i in val_ids)}")
print()

# Inspecionar estrutura de uma entrada
sample_entry = manifest_dict.get("1a1m", next(iter(rcsb_data)))
print("Exemplo de entrada no manifesto (keys):", list(sample_entry.keys()))
if "chains" in sample_entry and sample_entry["chains"]:
    print("  chains[0] keys:", list(sample_entry["chains"][0].keys()))
if "interfaces" in sample_entry and sample_entry["interfaces"]:
    print("  interfaces[0] keys:", list(sample_entry["interfaces"][0].keys()))

Entradas no manifesto RCSB: 216870
IDs de treino presentes   : 1082
IDs de val presentes      : 130

Exemplo de entrada no manifesto (keys): ['id', 'structure', 'chains', 'interfaces', 'affinity', 'md']
  chains[0] keys: ['chain_id', 'chain_name', 'mol_type', 'cluster_id', 'msa_id', 'template_id', 'num_residues', 'valid']
  interfaces[0] keys: ['chain_1', 'chain_2', 'valid']


## 7. Salvar Manifestos Filtrados por Split

In [7]:
train_manifest_raw = [s for s in rcsb_data if s["id"] in train_ids]
val_manifest_raw   = [s for s in rcsb_data if s["id"] in val_ids]

with open(MHC_DATA_DIR / "train_templated.json", "w") as f:
    json.dump(train_manifest_raw, f)

with open(MHC_DATA_DIR / "val_templated.json", "w") as f:
    json.dump(val_manifest_raw, f)

print(f"train_templated.json : {len(train_manifest_raw)} entradas")
print(f"val_templated.json   : {len(val_manifest_raw)} entradas")

train_templated.json : 1082 entradas
val_templated.json   : 130 entradas


## 8. Função de Leitura de Sequências do NPZ

In [22]:
def load_chain_sequences_from_npz(pdb_id: str) -> dict:
    """
    Carrega o .npz de uma estrutura e retorna dict:
        {chain_id (int): {'seq': str, 'name': str, 'len': int, 'msa_id': int}}
    """
    npz_path = RCSB_DIR / "structures" / f"{pdb_id}.npz"
    if not npz_path.exists():
        return {}

    npz       = np.load(npz_path, allow_pickle=True)
    chains    = npz["chains"]
    residues  = npz["residues"]
    result    = {}

    for ch in chains:
        chain_id  = ch[0]
        res_start = int(ch[1])
        res_end   = res_start + int(ch[2])
        res_block = residues[res_start:res_end]
        seq       = decode_sequence(res_block["res_type"])
        chain_name = str(ch[3]) if len(ch) > 3 else ""
        msa_id     = int(ch[4])     if len(ch) > 4 else -1

        result[chain_id] = {
            "seq":        seq,
            "name":       chain_name,
            "len":        len(seq),
            "msa_id":     msa_id,
        }
    return result


# Teste rápido com 1A1M
chains_1a1m = load_chain_sequences_from_npz("1a1m")
for cid, info in chains_1a1m.items():
    print(f"  chain_id={cid}  name={info['name']:<20}  len={info['len']:>4}  msa_id={info['msa_id']}")


  chain_id=A1  name=0                     len=   0  msa_id=0
  chain_id=B1  name=0                     len=   1  msa_id=1
  chain_id=C1  name=0                     len=   2  msa_id=2


## 9. Identificação de Cadeias MHC-α e Peptídeo

**Lógica (corrige o `KeyError: 'protein_chains'`):**

1. Carrega as sequências de todas as cadeias do `.npz`.
2. Para cada cadeia com comprimento ≥ `MHC_ALPHA_MIN_LEN`, calcula identidade
   contra `MHC_ALPHA_TEMPLATE_SEQ` (1A1M) via BLOSUM62.
3. A cadeia com maior identidade ≥ `MHC_IDENTITY_THRESHOLD` é o MHC-α.
4. O peptídeo é a cadeia cuja sequência coincide com `Peptide*` do CSV
   **e** tem comprimento ≤ 25.


In [23]:
def identify_mhc_and_peptide_chains(
    pdb_id: str,
    peptide_seq: str,
    chain_sequences: dict,
) -> tuple:
    """
    Retorna (mhc_alpha_chain_id, peptide_chain_id) ou (None, None) se não encontrar.

    Args:
        pdb_id          : ID da estrutura (para logs)
        peptide_seq     : sequência do peptídeo conforme TCR3d_data.csv
        chain_sequences : saída de load_chain_sequences_from_npz()

    Returns:
        (mhc_alpha_chain_id: int | None, peptide_chain_id: int | None)
    """
    best_mhc_id    = None
    best_mhc_score = 0.0
    peptide_id     = None

    for chain_id, info in chain_sequences.items():
        seq = info["seq"]

        # ── Identificar peptídeo ─────────────────────────────────────────
        if is_peptide_sequence(seq, peptide_seq):
            peptide_id = chain_id

        # ── Identificar MHC-α ────────────────────────────────────────────
        elif info["len"] >= MHC_ALPHA_MIN_LEN:
            score = sequence_identity(seq, MHC_ALPHA_TEMPLATE_SEQ)
            if score > best_mhc_score:
                best_mhc_score = score
                best_mhc_id    = chain_id

    if best_mhc_score < MHC_IDENTITY_THRESHOLD:
        best_mhc_id = None

    return best_mhc_id, peptide_id


# ── Teste com 1A1M ───────────────────────────────────────────────────────────
peptide_1a1m = "TPYDINQML"
mhc_id, pep_id = identify_mhc_and_peptide_chains("1a1m", peptide_1a1m, chains_1a1m)
print(f"1A1M → MHC-α chain_id={mhc_id}  |  Peptídeo chain_id={pep_id}")
if mhc_id is not None:
    print(f"  MHC-α seq (primeiros 40): {chains_1a1m[mhc_id]['seq'][:40]}...")
if pep_id is not None:
    print(f"  Peptídeo seq            : {chains_1a1m[pep_id]['seq']}")

1A1M → MHC-α chain_id=None  |  Peptídeo chain_id=None


## 10. Loop Principal — Extração e Filtragem do Subconjunto MHC

In [24]:
TARGET_DIR.mkdir(parents=True, exist_ok=True)
(TARGET_DIR / "msa").mkdir(exist_ok=True)
(TARGET_DIR / "structures").mkdir(exist_ok=True)

# Combina train + val para processar de uma vez
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
train_val    = train_val_df.to_dict(orient="records")

new_manifest    = []
bad_ids         = []
skipped_reasons = {}   # pdb_id → motivo do descarte

for sample in train_val:
    pdb_id      = sample["PDB ID"]
    peptide_seq = str(sample.get("Peptide*", "")).strip()

    # ── 1. ID no manifesto RCSB? ─────────────────────────────────────────
    if pdb_id not in manifest_dict:
        bad_ids.append(pdb_id)
        skipped_reasons[pdb_id] = "não encontrado no manifesto RCSB"
        continue

    sample_manifest = manifest_dict[pdb_id]

    # ── 2. Carregar sequências do NPZ ────────────────────────────────────
    chain_sequences = load_chain_sequences_from_npz(pdb_id)
    if not chain_sequences:
        bad_ids.append(pdb_id)
        skipped_reasons[pdb_id] = "NPZ não encontrado ou vazio"
        continue

    # ── 3. Identificar MHC-α e peptídeo via alinhamento ─────────────────
    mhc_chain_id, pep_chain_id = identify_mhc_and_peptide_chains(
        pdb_id, peptide_seq, chain_sequences
    )

    if mhc_chain_id is None or pep_chain_id is None:
        bad_ids.append(pdb_id)
        skipped_reasons[pdb_id] = (
            f"cadeia MHC-α não encontrada"  if mhc_chain_id is None else
            f"cadeia peptídeo não encontrada (seq={peptide_seq})"
        )
        continue

    valid_chain_ids = {mhc_chain_id, pep_chain_id}

    # ── 4. Copiar MSAs das cadeias válidas ───────────────────────────────
    for chain in sample_manifest.get("chains", []):
        msa_id = chain.get("msa_id", -1)
        if msa_id != -1:
            src = RCSB_MSA_DIR / f"{msa_id}.npz"
            dst = TARGET_DIR / "msa" / f"{msa_id}.npz"
            if src.exists() and not dst.exists():
                shutil.copy(src, dst)

    # ── 5. Marcar cadeias inválidas no manifesto ─────────────────────────
    for chain in sample_manifest.get("chains", []):
        if chain.get("chain_id") not in valid_chain_ids:
            chain["valid"] = False

    # ── 6. Validar interface MHC-α ↔ peptídeo ───────────────────────────
    n_correct_interfaces = 0
    for interface in sample_manifest.get("interfaces", []):
        c1 = interface.get("chain_1")
        c2 = interface.get("chain_2")
        if c1 in valid_chain_ids and c2 in valid_chain_ids:
            n_correct_interfaces += 1
        else:
            interface["valid"] = False

    if n_correct_interfaces != 1:
        bad_ids.append(pdb_id)
        skipped_reasons[pdb_id] = f"n_interfaces_corretas={n_correct_interfaces} (esperado 1)"
        continue

    # ── 7. Atualizar máscara no NPZ ──────────────────────────────────────
    npz_src  = RCSB_DIR / "structures" / f"{pdb_id}.npz"
    npz_dst  = TARGET_DIR / "structures" / f"{pdb_id}.npz"
    npz_dict = dict(np.load(npz_src, allow_pickle=True))

    for i in range(len(npz_dict["mask"])):
        npz_dict["mask"][i] = (i in valid_chain_ids)

    np.savez(str(npz_dst), **npz_dict)

    # ── 8. Adicionar ao novo manifesto ───────────────────────────────────
    # Enriquece com metadados do CSV
    sample_manifest["mhc_allele"]   = sample.get("MHC allele", "")
    sample_manifest["peptide_seq"]  = peptide_seq
    sample_manifest["mhc_chain_id"] = mhc_chain_id
    sample_manifest["pep_chain_id"] = pep_chain_id
    new_manifest.append(sample_manifest)


print(f"\n{'='*55}")
print(f"Processados  : {len(train_val)}")
print(f"Aceitos      : {len(new_manifest)}")
print(f"Descartados  : {len(bad_ids)}")
print(f"{'='*55}")
if skipped_reasons:
    from collections import Counter
    reason_counts = Counter(skipped_reasons.values())
    print("\nMotivos de descarte:")
    for reason, count in reason_counts.most_common():
        print(f"  {count:>4}x  {reason}")


Processados  : 1212
Aceitos      : 0
Descartados  : 1212

Motivos de descarte:
  1212x  cadeia MHC-α não encontrada


## 11. Salvar Manifesto Final e validation_ids.txt

In [ ]:
# Manifesto principal para o BoltzTrainingDataModule
with open(TARGET_DIR / "manifest.json", "w") as f:
    json.dump(new_manifest, f, indent=2, default=str)

# IDs de validação (uppercase, formato esperado pelo Boltz1)
val_set   = set(val_ids)
val_final = [s["id"].upper() for s in new_manifest if s["id"] in val_set]

with open(TARGET_DIR / "validation_ids.txt", "w") as f:
    f.write("\n".join(val_final))

print(f"Manifesto salvo : {TARGET_DIR}/manifest.json  ({len(new_manifest)} entradas)")
print(f"validation_ids  : {TARGET_DIR}/validation_ids.txt  ({len(val_final)} IDs)")

# Contagem por split
train_final = [s for s in new_manifest if s["id"] in set(train_ids)]
print(f"\nDistribuição no manifesto final:")
print(f"  train : {len(train_final)}")
print(f"  val   : {len(val_final)}")

## 12. Sanity Check — Inspeção de uma Estrutura

In [ ]:
if new_manifest:
    sample = new_manifest[0]
    pdb_id = sample["id"]
    print(f"Estrutura: {pdb_id.upper()}")
    print(f"  MHC allele     : {sample.get('mhc_allele', 'N/A')}")
    print(f"  Peptídeo       : {sample.get('peptide_seq', 'N/A')}")
    print(f"  MHC chain_id   : {sample.get('mhc_chain_id')}")
    print(f"  Pep chain_id   : {sample.get('pep_chain_id')}")
    print()

    # Verificar máscara no NPZ gerado
    npz_out = np.load(TARGET_DIR / "structures" / f"{pdb_id}.npz", allow_pickle=True)
    print(f"  mask no NPZ filtrado: {npz_out['mask']}")
    print()

    # Listar cadeias aceitas
    chains = npz_out["chains"]
    print("  Cadeias com mask=True:")
    for i, ch in enumerate(chains):
        if npz_out["mask"][i]:
            cname = str(ch["chain_name"]) if "chain_name" in ch.dtype.names else "?"
            rnum  = int(ch["res_num"])    if "res_num"    in ch.dtype.names else 0
            print(f"    chain_id={i}  name={cname:<20}  res_num={rnum}")

## 13. Resumo de QC — Distribuição de Comprimentos

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        import matplotlib.pyplot as plt

        mhc_lens = []
        pep_lens = []

        for s in new_manifest[:200]:   # amostra rápida
            seqs = load_chain_sequences_from_npz(s["id"])
            mhc_id_ = s.get("mhc_chain_id")
            pep_id_ = s.get("pep_chain_id")
            if mhc_id_ is not None and mhc_id_ in seqs:
                mhc_lens.append(seqs[mhc_id_]["len"])
            if pep_id_ is not None and pep_id_ in seqs:
                pep_lens.append(seqs[pep_id_]["len"])

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].hist(mhc_lens, bins=30, color="#4C72B0", edgecolor="white")
        axes[0].set_title("Comprimento MHC-α (nº resíduos)")
        axes[0].set_xlabel("Resíduos"); axes[0].set_ylabel("Frequência")

        axes[1].hist(pep_lens, bins=range(6, 20), color="#DD8452", edgecolor="white")
        axes[1].set_title("Comprimento Peptídeo (nº resíduos)")
        axes[1].set_xlabel("Resíduos"); axes[1].set_ylabel("Frequência")

        plt.suptitle(f"Distribuição de comprimentos — {len(mhc_lens)} estruturas amostradas")
        plt.tight_layout()
        plt.savefig(TARGET_DIR / "qc_length_distributions.png", dpi=150)
        plt.show()
        print("Gráfico salvo em:", TARGET_DIR / "qc_length_distributions.png")

        print(f"\nMHC-α: min={min(mhc_lens)}, max={max(mhc_lens)}, mediana={sorted(mhc_lens)[len(mhc_lens)//2]}")
        print(f"Peptídeo: min={min(pep_lens)}, max={max(pep_lens)}, mediana={sorted(pep_lens)[len(pep_lens)//2]}")

    except Exception as e:
        print(f"matplotlib não disponível ou erro: {e}")

## 14. Exportar Relatório QC em JSON (Data Contract)

In [ ]:
import datetime

qc_report = {
    "generated_at"   : datetime.datetime.now().isoformat(),
    "template_used"  : "1A1M",
    "identity_threshold": MHC_IDENTITY_THRESHOLD,
    "splits": {
        "train_cutoff" : str(TRAIN_CUTOFF.date()),
        "val_cutoff"   : str(VAL_CUTOFF.date()),
    },
    "counts": {
        "input_train_val" : len(train_val),
        "accepted"        : len(new_manifest),
        "rejected"        : len(bad_ids),
        "train_final"     : len(train_final),
        "val_final"       : len(val_final),
    },
    "rejection_reasons": dict(
        sorted(
            __import__("collections").Counter(skipped_reasons.values()).items(),
            key=lambda x: -x[1]
        )
    ),
    "bad_ids_sample"  : bad_ids[:20],
}

qc_path = TARGET_DIR / "qc_report.json"
with open(qc_path, "w") as f:
    json.dump(qc_report, f, indent=2)

print("QC report salvo em:", qc_path)
print(json.dumps(qc_report["counts"], indent=2))

## 15. Próximos Passos

Com o subconjunto gerado em `mhc_samples/`, o pipeline continua:

```bash
# Rodar finetuning com o manifesto filtrado
python scripts/train/train.py \
    data.train_manifest=mhc_samples/manifest.json \
    data.val_ids=mhc_samples/validation_ids.txt \
    data.target_dir=mhc_samples \
    data.msa_dir=mhc_samples/msa
```

**Checklist antes de iniciar o treino:**
- [ ] `qc_report.json` revisado — taxa de aceite ≥ 70%
- [ ] Histogramas de comprimento inspecionados (peptídeos 8–14 aa, MHC-α 80–400 aa)
- [ ] Pelo menos 1 estrutura aberta no PyMOL/ChimeraX para validação visual
- [ ] DockQ rodado em 3–5 amostras representativas (Enzo)
- [ ] `mhc_samples/` comprimido e transferido para o servidor A100

**Teach-back obrigatório na entrega** (Seção 2.3 do Action Plan):
explicar a decisão de usar BLOSUM62 com limiar de 30% de identidade.
